In [2]:
from new_module.dev_utils.utils import *

### Sentiment Control 결과

In [ ]:
# 원본 파일에서 length 별로 index가 어떻게 되는지 체크 
original = read_outputs('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set.jsonl')
edited_only_index = [int(x) for x in open('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_index.jsonl', 'r').read().split()]
edited_only = original.loc[edited_only_index, :].reset_index(drop=True)
len_12_index = edited_only[edited_only['seq_lengths'] == 12].index.tolist()
len_20_index = edited_only[edited_only['seq_lengths'] == 20].index.tolist()
len_50_index = edited_only[edited_only['seq_lengths'] == 50].index.tolist()


loc edit mlm

In [ ]:

# sbertscore 파일을 읽어와서 길이별로 sbertscore가 어떻게 다른지 확인
loc_edit_mlm_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/sentiment/positive_gpt2/kvi2h6uo/outputs_epsilon0.97.txt.3-results.txt.sbertscore', 
                   'sbertscore')

In [20]:
print(f'Mean BertScore (Length 12): {loc_edit_mlm_sbert[len_12_index].mean()}')
print(f'Mean BertScore (Length 20): {loc_edit_mlm_sbert[len_20_index].mean()}')
print(f'Mean BertScore (Length 50): {loc_edit_mlm_sbert[len_50_index].mean()}')

Mean BertScore (Length 12): 0.4179540899038083
Mean BertScore (Length 20): 0.4533534828270052
Mean BertScore (Length 50): 0.6261323625704207


In [22]:
print(f'Median BertScore (Length 12): {np.quantile(loc_edit_mlm_sbert[len_12_index],0.5)}')
print(f'Median BertScore (Length 20): {np.quantile(loc_edit_mlm_sbert[len_20_index],0.5)}')
print(f'Median BertScore (Length 50): {np.quantile(loc_edit_mlm_sbert[len_50_index],0.5)}')

Median BertScore (Length 12): 0.4056054651737213
Median BertScore (Length 20): 0.45544862747192383
Median BertScore (Length 50): 0.6388474106788635


In [ ]:

# sbertscore 파일을 읽어와서 길이별로 positive 결과가 어떻게 다른지 확인
loc_edit_mlm_senti = read_metric_file('/data/hyeryung/mucoco/outputs/sentiment/positive_gpt2/kvi2h6uo/outputs_epsilon0.97.txt.3-results.txt.sentiment_ext', 
                   'sentiment_ext')

In [28]:
print(f'Mean Positive % (Length 12): {loc_edit_mlm_senti[len_12_index].mean()}')
print(f'Mean Positive % (Length 20): {loc_edit_mlm_senti[len_20_index].mean()}')
print(f'Mean Positive % (Length 50): {loc_edit_mlm_senti[len_50_index].mean()}')

Mean Positive % (Length 12): 0.9084249084249084
Mean Positive % (Length 20): 0.8818897637795275
Mean Positive % (Length 50): 0.8406374501992032


loc edit llm

In [ ]:

# sbertscore 파일을 읽어와서 길이별로 sbertscore가 어떻게 다른지 확인
loc_edit_llm_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/dump/iter_loc_edit_qwen/final/16_pos_loc_edit_38144.jsonl-results.txt.sbertscore', 
                   'sbertscore')

In [ ]:
print(f'Mean BertScore (Length 12): {loc_edit_llm_sbert[len_12_index].mean()}')
print(f'Mean BertScore (Length 20): {loc_edit_llm_sbert[len_20_index].mean()}')
print(f'Mean BertScore (Length 50): {loc_edit_llm_sbert[len_50_index].mean()}')

Mean BertScore (Length 12): 0.6789290775978194
Mean BertScore (Length 20): 0.6139149231209207
Mean BertScore (Length 50): 0.6313791473756036


In [ ]:
print(f'Median BertScore (Length 12): {np.quantile(loc_edit_llm_sbert[len_12_index],0.5)}')
print(f'Median BertScore (Length 20): {np.quantile(loc_edit_llm_sbert[len_20_index],0.5)}')
print(f'Median BertScore (Length 50): {np.quantile(loc_edit_llm_sbert[len_50_index],0.5)}')

Median BertScore (Length 12): 0.7394208312034607
Median BertScore (Length 20): 0.6796927154064178
Median BertScore (Length 50): 0.6551200151443481


### NLI 결과
NLI 생성문에서 결과가 neutral이면 BertScore가 낮은지 확인

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct')
original = read_outputs('/data/hyeryung/mucoco/new_module/data/logical-consistency/anli-r2-test_prompt_4_below_consistent_threshold_3105.jsonl')
original['tokens'] = original['text'].apply(lambda x: tokenizer.tokenize(x))
original['seq_lengths'] = original['tokens'].apply(lambda x: len(x))

loc edit mlm : neutral 이면 낮음. 길이가 짧아도 낮음.

In [ ]:
### nli 결과 파일과 bertscore을 모두 읽어오기 
loc_edit_mlm_nli = read_metric_file('/data/hyeryung/mucoco/outputs/nli/ay8ohdbp/results_epsilon0.99-test.txt.nli', 'nli')
loc_edit_mlm_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/nli/ay8ohdbp/results_epsilon0.99-test.txt.sbertscore', 'sbertscore')

In [59]:
loc_edit_mlm_nli['sbertscore'] = loc_edit_mlm_sbert
loc_edit_mlm_nli['original_seq_lengths'] = original['seq_lengths']
loc_edit_mlm_nli['original_seq_lengths_bins'] = pd.cut(original['seq_lengths'], bins=[0, 10, 20, 30, 110])

In [60]:
loc_edit_mlm_nli.groupby('nli_class')['sbertscore'].mean()

nli_class
contradiction    0.705955
entail           0.746062
neutral          0.639200
Name: sbertscore, dtype: float64

In [11]:
# original seq 길이가 길어질 수록 같은 class 여도 bertscore이 높아짐
# 같은 original seq bin에서 class에 따라 entail > contradiction > neutral 순으로 bertscore이 높음
loc_edit_mlm_nli.groupby(['original_seq_lengths_bins', 'nli_class'])['sbertscore'].mean().reset_index()

,original_seq_lengths_bins,nli_class,sbertscore
0,"(0, 10]",contradiction,0.889978
1,"(0, 10]",entail,0.832271
2,"(0, 10]",neutral,0.855047
3,"(10, 20]",contradiction,0.920735
4,"(10, 20]",entail,0.863749
5,"(10, 20]",neutral,0.873593
6,"(20, 30]",contradiction,0.921984
7,"(20, 30]",entail,0.847504
8,"(20, 30]",neutral,0.884509
9,"(30, 110]",contradiction,0.922968


loc edit mlm 

num_edit_tokens = 1이면 sbertscore 높다. 원인이 num_edit_tokens가 줄면서 num_edit_tokens/seq length의 ratio가 줄었기 때문일까 아니면 결과 nli class에서 neutral의 비중이 줄었기 때문일까?

In [7]:
### nli 결과 파일과 bertscore을 모두 읽어오기 
loc_edit_mlm_nli = read_metric_file('/data/hyeryung/mucoco/outputs/nli/k0tprzvq/results_epsilon0.99-test.txt.nli', 'nli')
loc_edit_mlm_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/nli/k0tprzvq/results_epsilon0.99-test.txt.sbertscore', 'sbertscore')

In [8]:
loc_edit_mlm_nli['sbertscore'] = loc_edit_mlm_sbert
loc_edit_mlm_nli['original_seq_lengths'] = original['seq_lengths']
loc_edit_mlm_nli['original_seq_lengths_bins'] = pd.cut(original['seq_lengths'], bins=[0, 10, 20, 30, 110])

In [ ]:
# num_edit_tokens=7일 때보다 모든 class에서 bertscore이 높음
loc_edit_mlm_nli.groupby('nli_class')['sbertscore'].mean()

nli_class
contradiction    0.919414
entail           0.859661
neutral          0.881142
Name: sbertscore, dtype: float64

In [ ]:
# num_edit_tokens=7일 때와 slightly 다른건.. entail 할 때 bertscore가 낮다는것? 끝.
loc_edit_mlm_nli.groupby(['original_seq_lengths_bins', 'nli_class'])['sbertscore'].mean()

original_seq_lengths_bins  nli_class    
(0, 10]                    contradiction    0.889978
                           entail           0.832271
                           neutral          0.855047
(10, 20]                   contradiction    0.920735
                           entail           0.863749
                           neutral          0.873593
(20, 30]                   contradiction    0.921984
                           entail           0.847504
                           neutral          0.884509
(30, 110]                  contradiction    0.922968
                           entail           0.898675
                           neutral          0.935295
Name: sbertscore, dtype: float64

loc edit llm : 패턴 없음

In [62]:
### nli 결과 파일과 bertscore을 모두 읽어오기 

loc_edit_llm_nli = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/nli/6_nli_edited_38454.jsonl_total_0-results.txt.nli', 'nli')
loc_edit_llm_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/6_nli_edited_38454.jsonl_total_0-results.txt.sbertscore', 'sbertscore')

In [63]:
loc_edit_llm_nli['sbertscore'] = loc_edit_llm_sbert
loc_edit_llm_nli['original_seq_lengths'] = original['seq_lengths']
loc_edit_llm_nli['original_seq_lengths_bins'] = pd.cut(original['seq_lengths'], bins=[0, 10, 20, 30, 110])

In [64]:
loc_edit_llm_nli.groupby('nli_class')['sbertscore'].mean()

nli_class
contradiction    0.839988
entail           0.852316
neutral          0.869030
Name: sbertscore, dtype: float64

In [ ]:
# original seq 길이에 따른 패턴 없음
# 같은 original seq bin일 때 class에 따른 패턴 없음
loc_edit_llm_nli.groupby(['original_seq_lengths_bins', 'nli_class'])['sbertscore'].mean()

original_seq_lengths_bins  nli_class    
(0, 10]                    contradiction    0.822947
                           entail                NaN
                           neutral          0.878485
(10, 20]                   contradiction    0.856645
                           entail           0.892360
                           neutral          0.878336
(20, 30]                   contradiction    0.827646
                           entail           0.686214
                           neutral          0.850751
(30, 110]                  contradiction    0.772548
                           entail           0.867640
                           neutral          0.897055
Name: sbertscore, dtype: float64

llm edit wo locate : 패턴 없음

In [66]:
### nli 결과 파일과 bertscore을 모두 읽어오기 

llm_edit_nli = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/nli/9_nli_loc_edit_38546.jsonl-results.txt.nli', 'nli')
llm_edit_sbert = read_metric_file('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/9_nli_loc_edit_38546.jsonl-results.txt.sbertscore', 'sbertscore')

In [67]:
llm_edit_nli['sbertscore'] = llm_edit_sbert
llm_edit_nli['original_seq_lengths'] = original['seq_lengths']
llm_edit_nli['original_seq_lengths_bins'] = pd.cut(original['seq_lengths'], bins=[0, 10, 20, 30, 110])

In [68]:
llm_edit_nli.groupby('nli_class')['sbertscore'].mean()

nli_class
contradiction    0.802342
entail           0.886498
neutral          0.794597
Name: sbertscore, dtype: float64

In [69]:
# original seq 길이에 따른 패턴 없음
# 같은 original seq bin일 때 class에 따른 패턴 없음
llm_edit_nli.groupby(['original_seq_lengths_bins', 'nli_class'])['sbertscore'].mean()

original_seq_lengths_bins  nli_class    
(0, 10]                    contradiction    0.907226
                           entail                NaN
                           neutral          0.831641
(10, 20]                   contradiction    0.786899
                           entail           0.886498
                           neutral          0.795492
(20, 30]                   contradiction    0.814272
                           entail                NaN
                           neutral          0.786248
(30, 110]                  contradiction    0.720259
                           entail                NaN
                           neutral          0.813737
Name: sbertscore, dtype: float64